# 00. 설정 (settings)

**이 노트북이 하는 일**

프로젝트 전체에서 쓰는 설정을 **이 노트북 한 곳에만** 정의합니다: 경로, 대상 역, 가리기·더미 열 이름, 기간 분할, SARIMA 차수, 적합 옵션, 특보 열, 분석 기준값. 다른 노트북은 첫 셀에서 `%run ./00_settings.ipynb`로 이 설정을 불러옵니다. **설정을 바꿀 때는 이 노트북만 고칩니다.**

**입력 파일**

- 없음 (환경변수 `SUBWAY_PROC`, `SUBWAY_INTERIM`, `SUBWAY_FIG`, `SUBWAY_RESULTS`, `SUBWAY_EXTRA_PATH`로 경로를 바꿀 수 있음)

**출력 파일**

- 없음 (변수만 정의)

> 열 이름은 `data/processed/model_input_9stations.csv`의 실제 열 이름과 같습니다.

**Colab 준비**: Colab에서 실행하면 저장소를 clone하고 `notebooks/`로 이동합니다 (로컬이면 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')


**경로와 실행 환경**: 저장소 루트(`ROOT`)를 기준으로 상대경로를 정합니다. `notebooks/`에서 실행하면 한 단계 위가 루트입니다. `RAW_OK`는 원본 데이터(`rawdata/`)가 있는지 알려 줍니다 (없으면 01번 노트북은 처리된 파일을 읽기만 합니다).

In [2]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

# 추가 패키지 경로 (예: holidays, factor_analyzer를 따로 설치한 폴더)
for _p in os.environ.get('SUBWAY_EXTRA_PATH', '').split(os.pathsep):
    if _p:
        sys.path.append(_p)          # 뒤에 붙여서 기존 패키지가 우선

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()   # 저장소 루트
RAW = ROOT / 'rawdata'                                                          # 원본 (저장소에 없음)
PROC = Path(os.environ.get('SUBWAY_PROC', ROOT / 'data' / 'processed'))         # 처리된 데이터
INTERIM = Path(os.environ.get('SUBWAY_INTERIM', ROOT / 'data' / 'interim'))     # 중간 파일
FIG = Path(os.environ.get('SUBWAY_FIG', ROOT / 'figures'))                      # 그림
RESULTS = Path(os.environ.get('SUBWAY_RESULTS', ROOT / 'results'))              # 분석 결과 표
RAW_OK = RAW.is_dir() and any(p.is_file() and p.suffix in ('.csv', '.xlsx') for p in RAW.iterdir())   # 원본이 있는가

아래는 기존 `config.py`의 모든 설정입니다.

**대상 역 9개** (단독형 역: Employment 3개, Commercial & Leisure 3개, Residential 3개 순).

In [3]:
# 단독형 역 9개 (Employment / Commercial & Leisure / Residential 순)
STATIONS = ['시청', '역삼', '선릉', '신촌', '혜화', '신사', '서울대입구', '수유', '연신내']

**가리기용 0/1 열**: 1인 날은 모델에서 y를 NaN으로 가립니다 (날짜는 유지). 공휴일, 근로자의 날, 명절 ±2일, 코로나, 메르스, 파업, 시청 집회(시청역만), 특보 여부를 모르는 2014-01-01~07.

In [4]:
# 가리기용 0/1 열 (전 역 공통; rally_cityhall은 시청역만 값이 1, alert_unknown은 특보 여부를 모르는 2014-01-01~07)
MASK_COLS = ['holiday', 'labor_day', 'myeongjeol_pm2', 'covid', 'mers', 'strike', 'rally_cityhall', 'alert_unknown']

**계단 더미** (시행일부터 1, 전 역 공통): 요금 인상 3회와 기후동행카드.

In [5]:
# 계단 더미 (전 역 공통)
STEP_COLS = ['fare_2015', 'fare_2023', 'fare_2025', 'climate_card']

**계절 더미** (전 역 공통): 여름 휴가철(7월 마지막 월요일 ~ 8월 둘째 일요일).

In [6]:
# 계절 더미 (전 역 공통): 매년 7월 마지막 월요일 ~ 8월 둘째 일요일 = 1
SEASON_COLS = ['summer_vacation']

**역별 계단 더미**: 그 역에만 쓰는 더미 (서울대입구=신림선, 신사=신분당선 연장).

In [7]:
# 해당 역만 쓰는 계단 더미: 역 -> 열 이름
STATION_STEP = {'서울대입구': 'sillim_line', '신사': 'sinbundang_ext'}

**학습 끝날짜** (2014~2023 학습): `split` 열과 같은 기준으로 2014~2023은 train, 2024는 validation, 2025는 test입니다.

In [8]:
# 분할: 2014~2023 train, 2024 validation, 2025 test (split 열과 같은 기준)
TRAIN_END = '2023-12-31'

**검증 연도** (2024): 학습한 계수를 고정하고 하루씩 앞을 예측해 오차를 확인하는 데만 씁니다.

In [9]:
VALID = 2024

**테스트 연도** (2025): 07번 노트북 전까지 쓰지 않습니다.

In [10]:
TEST = 2025

**SARIMA 차수** `(p, d, q)`와 `(P, D, Q, s)` (확정안).

In [11]:
# SARIMA 차수: (p, d, q), (P, D, Q, s)
ORDER = (1, 0, 1)

**계절 차수** `(P, D, Q, s)`: 주기 7일(주간 계절성), 계절 차분 1회.

In [12]:
SEASONAL_ORDER = (1, 1, 1, 7)

**기상특보 열** (서울 전체 기준이라 9개 역이 같은 값): 폭염, 한파, 호우, 대설.

In [13]:
# 기상특보 열 (서울 전체 기준, 모든 역 같은 값; 한파 / 폭염 / 호우 / 대설)
ALERT_COLS = ['alert_heat', 'alert_cold', 'alert_rain', 'alert_snow']

**특보 후 가리는 일수**: 특보 해제 후 이 일수만큼 y를 가립니다 (04번 노트북에서 사용).

In [14]:
# 기상특보 발효 후 가리는 일수
ALERT_MASK_AFTER_DAYS = 7

**적합 방법**: 수렴 문제로 최적화 방법을 `lbfgs`로 명시합니다.

In [15]:
# 적합 설정 (수렴 문제로 최적화 방법과 최대 반복 횟수를 명시)
FIT_METHOD = 'lbfgs'

**최대 반복 횟수**: 최적화가 수렴하도록 1000회로 늘렸습니다.

In [16]:
FIT_MAXITER = 1000

**분석 설정 (SARIMAX 노트북에서 사용)**: 학습·검증 기간, 날씨 변수를 만드는 기준값(체감온도 33℃ 초과분, 최저기온 −12℃ 미만분), 날씨·특보 변수 이름과 시차, 차수 후보, 유형 순서, 병렬 작업 수, 미수렴 재시도 대상 역.

In [17]:
START = pd.Timestamp('2014-01-01')                                  # 분석 시작일
VALID_START, VALID_END = pd.Timestamp(f'{VALID}-01-01'), pd.Timestamp(f'{VALID}-12-31')   # 검증 연도(2024)의 시작·끝
HEAT_BASE, COLD_BASE = 33.0, -12.0                                  # heat_excess = max(0, 최고 체감온도 - 33), cold_excess = max(0, -12 - 최저기온)
WEATHER = ['heat_excess', 'cold_excess', 'rain', 'snow']            # ASOS 날씨 변수 4개
LAGS = [1, 2, 3]                                                    # 1~3일 전 값(lag)
WEATHER_ALL = WEATHER + [f'{c}_lag{k}' for c in WEATHER for k in LAGS]
ALERTS = list(ALERT_COLS)                                           # 특보 변수 4개 (alert_heat, alert_cold, alert_rain, alert_snow)
ALERT_ALL = ALERTS + [f'{c}_lag{k}' for c in ALERTS for k in LAGS]
TYPE_ORDER = ['Employment', 'Commercial & Leisure', 'Residential']  # 역 유형 순서
CANDIDATES = [(ORDER, SEASONAL_ORDER), ((2, 0, 1), SEASONAL_ORDER)]  # 차수 후보 (앞의 것이 확정안)
WORKERS = 6                                                         # 병렬 작업 수
RETRY_C_FROM_B = ['수유', '연신내']                                 # C 모형이 미수렴이었던 역: B 계수를 초기값으로 다시 시도

**이벤트 분석 설정 (02, 04번 노트북에서 사용)**: 특보 종류(한글 이름 → 열 이름), 계단 더미 시행일 앞뒤로 겹침을 확인하는 일수, 이벤트 수가 이보다 적으면 "참고용"으로 표시하는 기준.

In [18]:
EVENT_TYPES = {'폭염': 'alert_heat', '한파': 'alert_cold', '호우': 'alert_rain', '대설': 'alert_snow'}   # 특보 종류 -> 열 이름
EVENT_STEP_PM_DAYS = 30            # 계단 더미 시행일 앞뒤 이 일수 안에 이벤트가 걸리면 "겹침"
MIN_EVENTS = 10                    # 깨끗한 이벤트가 이보다 적으면 "참고용"으로 표시

**이벤트 기준선 분석 설정 (04번 노트북에서 사용)**: 기준선 예측 기간은 "이벤트 지속일수 + 해제 후 확인 일수"입니다. 충격·회복 판정 기준 k, m은 placebo(가짜 이벤트)의 오판정 비율로 정합니다.

In [19]:
EVENT_POST_DAYS = 14               # 해제 후 회복을 보는 일수 (기준선 예측 기간 = 지속일수 + 14일, 14일을 넘는 이벤트는 long_event)
BAND_ALPHA = 0.05                  # 예측구간 유의수준 (95% 구간)
KM_CANDIDATES = [1, 2, 3]          # 충격 판정 일수 k(구간 아래인 날이 k일 이상)와 회복 판정 연속 일수 m의 후보
FALSE_RATE_MAX = 0.05              # placebo에서 허용하는 오판정 비율 (5%)
PLACEBO_N = 300                    # 가짜 시작일 개수
PLACEBO_SEED = 42                  # 무작위 추출 시드
EVENT_MIN_HISTORY = 28             # 기준선에 필요한 시작 전 관측일 수(y가 가려지지 않은 날). 이보다 적으면 기준선을 믿을 수 없어 제외

**설정 요약**: 위에서 정의한 주요 설정을 표로 확인합니다.

In [20]:
pd.DataFrame({'설정': ['대상 역(STATIONS)', '가리기 열(MASK_COLS)', '계단 더미(STEP_COLS)', '계절 더미(SEASON_COLS)', '역별 더미(STATION_STEP)', '특보 열(ALERT_COLS)',
                      '학습 끝(TRAIN_END) / 검증 / 테스트', '차수(ORDER) / 계절 차수(SEASONAL_ORDER)', '적합(FIT_METHOD, FIT_MAXITER)', '특보 후 가리는 일수(ALERT_MASK_AFTER_DAYS)',
                      '차수 후보(CANDIDATES)', '폭염·한파 기준(HEAT_BASE, COLD_BASE)', '날씨 시차(LAGS)', '원본 있음(RAW_OK)', '이벤트 종류(EVENT_TYPES)', '계단 더미 겹침 일수 / 최소 이벤트 수', '해제 후 확인 일수 / 구간 유의수준', 'k, m 후보 / 오판정 허용 / placebo 수 / 시드', '기준선 최소 관측일(EVENT_MIN_HISTORY)'],
              '값': [STATIONS, MASK_COLS, STEP_COLS, SEASON_COLS, STATION_STEP, ALERT_COLS,
                    f'{TRAIN_END} / {VALID} / {TEST}', f'{ORDER} / {SEASONAL_ORDER}', f'{FIT_METHOD}, {FIT_MAXITER}', ALERT_MASK_AFTER_DAYS,
                    CANDIDATES, f'{HEAT_BASE}, {COLD_BASE}', LAGS, RAW_OK, EVENT_TYPES, f'{EVENT_STEP_PM_DAYS} / {MIN_EVENTS}', f'{EVENT_POST_DAYS} / {BAND_ALPHA}', f'{KM_CANDIDATES} / {FALSE_RATE_MAX} / {PLACEBO_N} / {PLACEBO_SEED}', EVENT_MIN_HISTORY]})

,설정,값
0,대상 역(STATIONS),"[시청, 역삼, 선릉, 신촌, 혜화, 신사, 서울대입구, 수유, 연신내]"
1,가리기 열(MASK_COLS),"[holiday, labor_day, myeongjeol_pm2, covid, me..."
2,계단 더미(STEP_COLS),"[fare_2015, fare_2023, fare_2025, climate_card]"
3,계절 더미(SEASON_COLS),[summer_vacation]
4,역별 더미(STATION_STEP),"{'서울대입구': 'sillim_line', '신사': 'sinbundang_ext'}"
5,특보 열(ALERT_COLS),"[alert_heat, alert_cold, alert_rain, alert_snow]"
6,학습 끝(TRAIN_END) / 검증 / 테스트,2023-12-31 / 2024 / 2025
7,차수(ORDER) / 계절 차수(SEASONAL_ORDER),"(1, 0, 1) / (1, 1, 1, 7)"
8,"적합(FIT_METHOD, FIT_MAXITER)","lbfgs, 1000"
9,특보 후 가리는 일수(ALERT_MASK_AFTER_DAYS),7
